In [ ]:
# =====================================================================
# CELL 1 — Mount Drive + check GPU
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# =====================================================================
# CELL 2 — Install the ORIGINAL LLaVA repo (Math-LLaVA is saved in this
# repo's native checkpoint format, not the HF transformers Llava format —
# that mismatch is what caused the CUDA assert last time).
#
# NOTE: We install with --no-deps because LLaVA's pyproject.toml pins
# torch==2.1.2, which no longer has a matching wheel on Colab and makes
# the whole install silently fail (llava then only "works" because we're
# sitting in its directory, running against Colab's much newer, INCOMPATIBLE
# transformers). Instead we install llava itself with --no-deps, then pin
# the exact package versions LLaVA's code expects.
# =====================================================================
!git clone https://github.com/haotian-liu/LLaVA.git
%cd LLaVA
!pip install -e . --no-deps -q
!pip install transformers==4.37.2 tokenizers==0.15.1 accelerate==0.21.0 \
    sentencepiece==0.1.99 einops==0.6.1 einops-exts==0.0.4 timm==0.6.13 \
    peft bitsandbytes datasets tqdm -q


Cloning into 'LLaVA'...
remote: Enumerating objects: 2297, done.
remote: Total 2297 (delta 0), reused 0 (delta 0), pack-reused 2297 (from 1)
Receiving objects: 100% (2297/2297), 13.71 MiB | 33.91 MiB/s, done.
Resolving deltas: 100% (1405/1405), done.
/content/LLaVA
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 37.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

# =====================================================================
# CELL 3 — Imports and paths
# =====================================================================
import os
os.environ["HF_HOME"] = "/content/hf_cache"

import json
import re
from tqdm import tqdm
from datasets import load_dataset

from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, tokenizer_image_token, process_images
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

DRIVE_OUT_DIR = "/content/drive/MyDrive/mathvista_results"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(DRIVE_OUT_DIR, "math_llava_13b_results.json")
CHECKPOINT_EVERY = 20


Running on device: cuda


In [ ]:
# =====================================================================
# CELL 4 — Parsing logic (unchanged from your original)
# =====================================================================
def clean_free_form(text):
    if not isinstance(text, str):
        return str(text)
    text = text.strip()
    text_lower = text.lower()

    prefixes = [
        "the answer is", "therefore, the answer is", "so the answer is",
        "the value is", "answer is", "value is", "equals", "it is"
    ]
    for prefix in prefixes:
        if text_lower.startswith(prefix):
            text = text[len(prefix):].strip()
            text_lower = text.lower()

    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match:
        text = match.group(1).strip()

    text = text.rstrip('.!?*, ')
    return text

def get_most_similar(extraction, choices):
    distances = []
    for choice in choices:
        overlap = len(set(extraction.lower()) & set(choice.lower()))
        distances.append(-overlap)
    ind = distances.index(min(distances))
    return choices[ind]

def normalize_extracted_answer(extraction, choices, question_type, answer_type, precision=2):
    if isinstance(extraction, str):
        extraction = extraction.strip()
    else:
        try:
            extraction = str(extraction)
        except:
            extraction = ""

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        if len(letter) > 0:
            extraction = letter[0].upper()
        else:
            letter_match = re.findall(r'^[a-zA-Z]$', extraction)
            if len(letter_match) > 0:
                extraction = letter_match[0].upper()
            else:
                for prefix in ["option ", "answer is "]:
                    if extraction.lower().startswith(prefix):
                        cand = extraction[len(prefix):].strip()
                        if len(cand) == 1 and cand.isalpha():
                            extraction = cand.upper()
                            break

        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            ind = options.index(extraction)
            extraction = choices[ind]
        else:
            cleaned = clean_free_form(extraction)
            extraction = get_most_similar(cleaned, choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            if len(numbers) > 0:
                extraction = numbers[0]
            else:
                extraction = cleaned
        else:
            extraction = cleaned

    if answer_type == 'integer':
        try:
            extraction = str(int(float(extraction)))
        except:
            pass
    elif answer_type == 'float':
        try:
            extraction = str(round(float(extraction), precision))
        except:
            pass

    return extraction

def is_correct(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip():
        return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5:
                return 1
        except:
            pass
    return 0

In [ ]:
# =====================================================================
# CELL 5 — Load dataset and model (native LLaVA loader, 4-bit)
# =====================================================================
print("\nLoading MathVista testmini dataset...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
print(f"Loaded {len(mathvista)} test samples.")

model_path = "Zhiqiang007/Math-LLaVA"
model_name = get_model_name_from_path(model_path)
print(f"\nLoading model: {model_path} (4-bit, native LLaVA loader)...")

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,          # Math-LLaVA is a full fine-tune, not a LoRA delta
    model_name=model_name,
    load_4bit=True,
    device_map="auto"
)
print("Model loaded successfully.")

conv_mode = "llava_v1"  # LLaVA-1.5 (Vicuna-based) conversation template


Loading MathVista testmini dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…):   0%|          | 0.00/142M [00:00<?, ?B/s]

data/test-00000-of-00002-6b81bd7f7e2065e(…):   0%|          | 0.00/358M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Some weights of the model checkpoint at Zhiqiang007/Math-LLaVA were not used when initializing LlavaLlamaForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.l

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# =====================================================================
# CELL 6 — Evaluation loop WITH resume support
# =====================================================================
limit = 1000  # try e.g. 20-50 first to confirm correctness/speed before the full run

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)
    done_pids = {r["pid"] for r in results}
    print(f"Resuming: found {len(results)} existing results, will skip those.")
else:
    results = []
    done_pids = set()

print(f"\nStarting evaluation on {limit} samples...")
for i in tqdm(range(min(limit, len(mathvista)))):
    sample = mathvista[i]

    if sample["pid"] in done_pids:
        continue

    question = sample["question"]
    image = sample["decoded_image"].convert("RGB")

    conv = conv_templates[conv_mode].copy()
    inp = DEFAULT_IMAGE_TOKEN + '\n' + question
    conv.append_message(conv.roles[0], inp)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    image_tensor = process_images([image], image_processor, model.config)
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            do_sample=False,
            max_new_tokens=256,
            use_cache=True
        )

    raw_answer = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

    choices = sample["choices"]
    question_type = sample["question_type"]
    answer_type = sample["answer_type"]
    gt_answer = sample["answer"]

    parsed_answer = normalize_extracted_answer(raw_answer, choices, question_type, answer_type)
    correct = is_correct(parsed_answer, gt_answer, answer_type)

    # MathVista has no top-level "category" field. The 7 reasoning categories
    # (geometry, arithmetic, algebra, logic, numeric, scientific, statistical)
    # live in sample["metadata"]["skills"], which is a LIST — a question can
    # require more than one skill and counts toward each.
    skills = sample["metadata"].get("skills", [])

    results.append({
        "pid": sample["pid"],
        "question": question,
        "raw_response": raw_answer,
        "parsed_response": parsed_answer,
        "ground_truth": gt_answer,
        "skills": skills,
        "correct": correct
    })

    if len(results) % CHECKPOINT_EVERY == 0:
        with open(RESULTS_PATH, "w") as f:
            json.dump(results, f, indent=4)

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=4)
print(f"\nResults saved to Drive: {RESULTS_PATH}")


Starting evaluation on 1000 samples...


100%|██████████| 1000/1000 [50:01<00:00,  3.00s/it]


Results saved to Drive: /content/drive/MyDrive/mathvista_results/math_llava_13b_results.json


In [ ]:
# =====================================================================
# CELL 7 — Metrics
# =====================================================================
# Maps the raw skill strings in metadata["skills"] to short category labels.
# A question can require multiple skills, so it can count toward multiple
# categories at once (this matches the official MathVista evaluation).
skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    correct = res["correct"]

    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1

    for skill in res.get("skills", []):
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*55)
print(f"{'Evaluation Summary':^55}")
print("="*55)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-"*55)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*55)


                  Evaluation Summary                   
Category             | Correct    | Total      | Accuracy  
-------------------------------------------------------
All                  | 365        | 1000       | 36.50%
Geometry             | 76         | 239        | 31.80%
Arithmetic           | 137        | 353        | 38.81%
Algebra              | 84         | 281        | 29.89%
Logic                | 7          | 37         | 18.92%
Numeric              | 52         | 144        | 36.11%
Scientific           | 37         | 122        | 30.33%
Statistical          | 118        | 301        | 39.20%
